In [39]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

In [40]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [41]:
# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": -8.9667,
	"longitude": -72.7833,
	"start_date": (pd.Timestamp("today", tz="UTC") - pd.DateOffset(years=1)).strftime("%Y-%m-%d"),
	"end_date": pd.Timestamp("today", tz="UTC").strftime("%Y-%m-%d"),
	"daily": ["temperature_2m_mean", "temperature_2m_max", "temperature_2m_min", "precipitation_sum", "daylight_duration"],
}
responses = openmeteo.weather_api(url, params = params)

In [42]:
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")


Coordinates: -8.963092803955078°N -72.80899047851562°E
Elevation: 243.0 m asl
Timezone difference to GMT+0: 0s


In [43]:
# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_temperature_2m_mean = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_max = daily.Variables(1).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(2).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(3).ValuesAsNumpy()
daily_daylight_duration = daily.Variables(4).ValuesAsNumpy()

In [44]:
daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

In [45]:
daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["precipitation_sum"] = daily_precipitation_sum
daily_data["daylight_duration"] = daily_daylight_duration

In [46]:
daily_dataframe = pd.DataFrame(data = daily_data)

In [47]:
daily_dataframe["date"] = daily_dataframe["date"].dt.date

In [48]:
daily_dataframe

,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,daylight_duration
0,2025-04-03,23.829168,27.000000,22.100000,33.099998,43188.906250
1,2025-04-04,24.264580,28.500000,21.700001,4.400000,43159.847656
2,2025-04-05,23.572916,25.650000,21.650000,32.800003,43130.859375
3,2025-04-06,24.333334,29.500000,20.400000,0.400000,43101.953125
4,2025-04-07,25.133337,31.150000,20.250000,0.000000,43073.164062
...,...,...,...,...,...,...
361,2026-03-30,23.970833,28.299999,21.400000,3.400000,43312.320312
362,2026-03-31,24.622917,29.950001,21.850000,4.000000,43282.824219
363,2026-04-01,25.216669,30.900000,21.950001,2.700000,43253.390625
364,2026-04-02,25.077082,29.600000,22.750000,17.100000,43224.031250


In [49]:
daily_dataframe.to_csv("../../data/landing/weather_data.csv", index=False)